# AutoAssess — Train YOLOv8-seg on VehiDE (Colab)

Trains the damage-detection model on the **VehiDE** dataset (7 structural/cosmetic damage classes — scratch, dent, torn, broken_lamp, missing_part, punctured, shattered_glass) instead of CarDD's 6, to cover damage types CarDD has no equivalent for.

**Before running:** Runtime -> Change runtime type -> select a GPU (T4 is fine).

**You'll need a Kaggle API token.** Get one at kaggle.com -> Settings -> API -> Create New Token (downloads `kaggle.json`). Add it as a Colab secret (key icon in the left sidebar) named `KAGGLE_JSON` containing the full file contents, or upload `kaggle.json` when prompted in the cell below.

## 1. Clone the repo and install dependencies

In [ ]:
!rm -rf /content/autoassess
!git clone https://github.com/DhyeyTandel/autoassess.git /content/autoassess
%cd /content/autoassess
!pip install -q -e .
!pip install -q kaggle
import os
os.makedirs('/content/autoassess/runs', exist_ok=True)
print("Verifying scripts directory:")
!ls -F /content/autoassess/scripts

### Verify `autoassess` Repository Contents

Let's check if the `autoassess` repository was cloned correctly and contains the expected `scripts` and `configs` directories, as well as the script files themselves.

In [ ]:
import os

repo_root = '/content/autoassess'
scripts_dir = os.path.join(repo_root, 'scripts')
configs_dir = os.path.join(repo_root, 'configs')

print(f"Listing contents of {repo_root}:")
!ls -F {repo_root}

print(f"\nListing contents of {scripts_dir}:")
!ls -F {scripts_dir}

print(f"\nListing contents of {configs_dir}:")
!ls -F {configs_dir}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/autoassess_runs
!sed -i 's|^  output_root:.*|  output_root: "/content/drive/MyDrive/autoassess_runs"|' /content/autoassess/configs/base.yaml
!grep output_root /content/autoassess/configs/base.yaml

## 2. Kaggle credentials

Tries a Colab secret named `KAGGLE_JSON` first; falls back to a manual file upload if the secret isn't set.

In [ ]:
import json
import os
from pathlib import Path
from google.colab import files

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
kaggle_json_path = kaggle_dir / "kaggle.json"

wrote_from_secret = False
try:
    from google.colab import userdata
    raw = userdata.get("KAGGLE_JSON")
    if raw:
        # check if it is raw string or path
        try:
            json.loads(raw)
            kaggle_json_path.write_text(raw)
        except:
            # If it's just the key, we'd need username too, so we fallback
            raise ValueError("Secret is not a valid JSON string")
        wrote_from_secret = True
        print("Wrote ~/.kaggle/kaggle.json from Colab secret.")
except Exception as e:
    print(f"Colab secret not used: {e}")

if not wrote_from_secret:
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    if uploaded:
        fname = next(iter(uploaded))
        kaggle_json_path.write_bytes(uploaded[fname])
        # Also keep a copy in /content for safety
        with open('/content/kaggle.json', 'wb') as f:
            f.write(uploaded[fname])

os.chmod(kaggle_json_path, 0o600)
print("Kaggle credentials ready at", kaggle_json_path)

## 3. Download + convert VehiDE

Same scripts used locally — `scripts/download_vehide.py` (Kaggle API) and `autoassess-convert-vehide` (VIA JSON -> YOLO-seg format).

In [ ]:
import os, json, pathlib, shutil

kaggle_json = pathlib.Path.home() / '.kaggle' / 'kaggle.json'
content_json = pathlib.Path('/content/kaggle.json')

# Ensure credentials are in the right place
if content_json.exists() and not kaggle_json.exists():
    (pathlib.Path.home() / '.kaggle').mkdir(exist_ok=True)
    shutil.copy(content_json, kaggle_json)
    os.chmod(kaggle_json, 0o600)

if kaggle_json.exists():
    with open(kaggle_json, 'r') as f:
        creds = json.load(f)
    os.environ['KAGGLE_USERNAME'] = creds['username']
    os.environ['KAGGLE_KEY'] = creds['key']
    print(f'API credentials set for user: {creds["username"]}')

    # Run download script
    !python /content/autoassess/scripts/download_vehide.py --config /content/autoassess/configs/base.yaml
else:
    print('Error: kaggle.json not found. Please run cell vkANDH5_G4Cd first.')

In [ ]:
import os
# Use absolute path verified in the repository structure
script_path = "/content/autoassess/src/autoassess/data/convert_vehide.py"
config_path = "/content/autoassess/configs/base.yaml"
dataset_path = "/content/autoassess/configs/vehide.yaml"

if os.path.exists(script_path):
    !python {script_path} --config {config_path} --dataset-config {dataset_path}
else:
    print(f"Error: {script_path} not found.")

### Detailed Inspection of `autoassess-convert-vehide` Output

Since the previous check indicated missing validation data, let's perform a more detailed inspection of the directory structure and contents immediately after the `autoassess-convert-vehide` command (`t_NbfjCMG4Cd`) to understand what was actually generated.

In [ ]:
import os
from pathlib import Path

yolo_dataset_root = Path('/content/autoassess/data/processed/vehide')

print(f"Inspecting YOLO dataset root: {yolo_dataset_root}")

if not yolo_dataset_root.exists():
    print(f"  ❌ The root directory {yolo_dataset_root} does not exist at all.")
else:
    print(f"  ✅ Root directory exists.")
    print("  Contents of root directory:")
    for item in yolo_dataset_root.iterdir():
        if item.is_dir():
            print(f"    [DIR] {item.name}/")
            # List contents of subdirectories like 'images' and 'labels'
            if item.name in ['images', 'labels']:
                for sub_item in item.iterdir():
                    if sub_item.is_dir():
                        print(f"      [SUBDIR] {sub_item.name}/")
                        # Count files in the split directories (train, val, test)
                        file_count = len(list(sub_item.iterdir()))
                        print(f"        Contains {file_count} files.")
                    else:
                        print(f"      [FILE] {sub_item.name}")
            else:
                # For other directories, just list their presence
                pass
        else:
            print(f"    [FILE] {item.name}")


### Debugging the IndexError: Check Processed Data

The `IndexError` often occurs when the model tries to evaluate a dataset that is empty or has issues. Let's verify that the processed validation images and labels directories contain files, as expected after the data conversion step.

In [ ]:
import os
from pathlib import Path

processed_data_root = Path('/content/autoassess/data/processed/vehide')
val_images_dir = processed_data_root / 'images' / 'val'
val_labels_dir = processed_data_root / 'labels' / 'val'

print(f"Checking validation images directory: {val_images_dir}")
if val_images_dir.exists() and any(val_images_dir.iterdir()):
    print(f"  ✅ Found {len(list(val_images_dir.iterdir()))} files in {val_images_dir}")
else:
    print(f"  ❌ Directory is empty or does not exist: {val_images_dir}")

print(f"Checking validation labels directory: {val_labels_dir}")
if val_labels_dir.exists() and any(val_labels_dir.iterdir()):
    print(f"  ✅ Found {len(list(val_labels_dir.iterdir()))} files in {val_labels_dir}")
else:
    print(f"  ❌ Directory is empty or does not exist: {val_labels_dir}")

# Optionally, list a few files to confirm structure
if val_images_dir.exists():
    print('\nFirst 5 validation images:')
    for i, file in enumerate(val_images_dir.iterdir()):
        if i >= 5: break
        print(f"- {file.name}")

if val_labels_dir.exists():
    print('\nFirst 5 validation labels:')
    for i, file in enumerate(val_labels_dir.iterdir()):
        if i >= 5: break
        print(f"- {file.name}")

If the above check shows that either the validation images or labels directories are empty or missing files, that would explain the `IndexError`. You would then need to investigate why the `autoassess-convert-vehide` command (`t_NbfjCMG4Cd`) didn't produce the expected output.

If the directories are populated, the issue might be more subtle, possibly related to:
- The content of the label files (e.g., all labels are empty or invalid).
- A configuration mismatch that causes the model to not detect any objects during evaluation.
- An issue within the `autoassess` library's evaluation logic itself for this specific dataset.

For a quick test, you could try temporarily skipping the final COCO evaluation step in the `autoassess-train-yolo` command if the library allows it, or reducing the number of epochs to see if the error occurs earlier.

## 4. Train YOLOv8-seg on VehiDE

Uses the repo's own defaults (`yolov8s-seg`, 50 epochs, batch 8, imgsz 640) — same training script (`train_yolo.py`) as the CarDD/parts models, just pointed at `configs/vehide.yaml`. `device 0` is Colab's GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
!autoassess-train-yolo \
  --dataset-config configs/vehide.yaml \
  --model yolov8s-seg.pt \
  --epochs 36 \
  --batch 16 \
  --imgsz 640 \
  --device 0 \
  --name vehide_seg_v1

In [ ]:
# RESUME ONLY. If the runtime died mid-training: rerun cells 1-3 + the Drive cell, then run THIS instead of the cell above.
!autoassess-train-yolo --dataset-config configs/vehide.yaml --epochs 36 --batch 16 --imgsz 640 --device 0 --name vehide_seg_v1 --resume

## 5. Check results

In [ ]:
import os
from pathlib import Path

# The training likely saved to /content/autoassess/runs/vehide_seg_v1
# Let's try to locate the actual runs directory
possible_paths = [
    Path("/content/drive/MyDrive/autoassess_runs/vehide_seg_v1"),
    Path("/content/autoassess/runs/vehide_seg_v1")
]

run_dir = None
for p in possible_paths:
    if p.exists():
        run_dir = p
        break

if run_dir:
    print(f"Found directory at: {run_dir.absolute()}")
    print(f"\nContents:")
    for item in run_dir.iterdir():
        if item.is_dir():
            print(f" [DIR]  {item.name}/")
            for sub in item.iterdir():
                print(f"        - {sub.name}")
        else:
            print(f" [FILE] {item.name}")
else:
    print("Could not find the runs directory. Checking current working directory...")
    print(f"CWD: {os.getcwd()}")
    print("Available folders here:", [d for d in os.listdir('.') if os.path.isdir(d)])

## 6. Package weights + metrics for download

Zips `runs/vehide_seg_v1/weights/{best,last}.pt` and `metrics.json` — download this and extract into `runs/vehide_seg_v1/` in your local repo so eval scripts and `app.py` can pick it up.

In [ ]:
R = "/content/drive/MyDrive/autoassess_runs/vehide_seg_v1"
import shutil

shutil.make_archive("/content/vehide_seg_v1_results", "zip", R)
print("Packaged: vehide_seg_v1_results.zip")

from google.colab import files
files.download("/content/vehide_seg_v1_results.zip")

### Troubleshooting Steps

First, please ensure that you have re-run cell `85Kaz5rMG4Cc` to clone the `autoassess` repository and install dependencies. Then, run the cell below to verify that the repository contents are accessible.

In [ ]:
import os

repo_root = '/content/autoassess'
scripts_dir = os.path.join(repo_root, 'scripts')
configs_dir = os.path.join(repo_root, 'configs')

print(f"Listing contents of {repo_root}:")
if os.path.exists(repo_root):
    !ls -F {repo_root}
else:
    print(f"❌ Repository root '{repo_root}' not found. Please re-run cell 85Kaz5rMG4Cc.")

if os.path.exists(scripts_dir):
    print(f"\nListing contents of {scripts_dir}:")
    !ls -F {scripts_dir}
else:
    print(f"❌ Scripts directory '{scripts_dir}' not found.")

if os.path.exists(configs_dir):
    print(f"\nListing contents of {configs_dir}:")
    !ls -F {configs_dir}
else:
    print(f"❌ Configs directory '{configs_dir}' not found.")


If the above cell shows that the `autoassess` directory and its contents are now visible, the next critical step is to ensure your Kaggle credentials are correctly set up. You need to make sure the `KAGGLE_JSON` secret is configured in Colab, or manually upload the `kaggle.json` file when prompted by cell `vkANDH5_G4Cd`. After verifying your credentials, re-run cell `vkANDH5_G4Cd` and then cell `AAaD3VdtG4Cd`.

Run the following cell to check if Kaggle credentials are now available.

In [ ]:
import os
from pathlib import Path

kaggle_json_path = Path.home() / ".kaggle" / "kaggle.json"

if kaggle_json_path.exists():
    print(f"✅ Kaggle credentials found at: {kaggle_json_path}")
    if 'KAGGLE_USERNAME' in os.environ and 'KAGGLE_KEY' in os.environ:
        print(f"✅ Kaggle environment variables are set.")
    else:
        print("⚠️ Kaggle credentials file found, but environment variables might not be set. Please ensure cell `AAaD3VdtG4Cd` is re-run after `vkANDH5_G4Cd`.")
else:
    print(f"❌ Kaggle credentials NOT found at: {kaggle_json_path}. Please set your `KAGGLE_JSON` secret or re-upload your `kaggle.json` file in cell `vkANDH5_G4Cd`.")


Once both the repository and Kaggle credentials are confirmed to be working, proceed to re-run cells `AAaD3VdtG4Cd` (Kaggle download prep) and `t_NbfjCMG4Cd` (VehiDE conversion). Then, verify the processed data using cells `37b1b103` and `a41b88ba`. If the data is correctly processed, you can then re-run the training cell `z0XB-egrG4Ce`. If successful, the final `shutil.make_archive` command in `gn5gdr5AG4Ce` should then work as expected.